In [3]:
from skimage import data
from skimage.restoration import estimate_sigma


In [1]:
#!/usr/bin/env python3
import os
import glob
import re
import numpy as np
import h5py
from skimage.restoration import estimate_sigma

from bm3d import bm3d
from bm4d import bm4d

# ==========================================
# 0. CONFIGURATION 
# ==========================================
base_dir = "/pscratch/sd/k/kberard/SCGSR/Data/diamond_1x1x1_bfd/density_data/vmc_J2"
ref_path = os.path.join(base_dir, "density_tot_ref_mean.h5")
dft_path = '/global/u2/k/kberard/SCGSR/Research/Diamond/Data/density_tot_ref.h5'

# ==========================================
# 1. UNIFIED MATH & UTILITY FUNCTIONS
# ==========================================
def D_JS(p1, p2, tol=1e-16):
    """Calculates Jensen-Shannon Divergence."""
    p1 = p1 / (np.sum(p1) + 1e-16)
    p2 = p2 / (np.sum(p2) + 1e-16)
    pm = (p1 + p2) / 2
    p1_nonzero, p2_nonzero = np.abs(p1) > tol, np.abs(p2) > tol
    
    d = 0.5 * (
        (np.abs(p1[p1_nonzero]) * np.log(np.abs(p1[p1_nonzero]) / np.abs(pm[p1_nonzero]))).sum() + 
        (np.abs(p2[p2_nonzero]) * np.log(np.abs(p2[p2_nonzero]) / np.abs(pm[p2_nonzero]))).sum()
    )
    return d / np.log(2)

def transform(density, density_ref, transform_type='residual_noise'):
    if transform_type == 'residual_noise':
        return (density - density_ref) / (np.sqrt(np.abs(density_ref)) + 1e-12)
    return density

def inverse_transform(density_trans, density_ref, transform_type='residual_noise'):
    if transform_type == 'residual_noise':
        return density_ref + (np.sqrt(np.abs(density_ref)) * density_trans)
    return density_trans

def encode_voxel_to_rgb_global(vol_3d):
    v_min, v_max = float(vol_3d.min()), float(vol_3d.max())
    if v_max == v_min: v_max = v_min + 1e-6
    normed = (vol_3d - v_min) / (v_max - v_min)
    return np.stack([normed]*3, axis=-1).astype(np.float32), v_min, v_max

def decode_rgb_to_voxel_global(rgb_volume, v_min, v_max):
    return rgb_volume[:, :, :, 0] * (v_max - v_min) + v_min

def evaluate_and_enforce(denoised_d, ref_d):
    denoised_d = np.maximum(denoised_d, 0.0)
    denoised_d = denoised_d * (8.0 / (np.sum(denoised_d) + 1e-16))
    return D_JS(denoised_d, ref_d)

# ==========================================
# 2. INFERENCE DISPATCHER
# ==========================================
def run_bm_with_sigma(test_d, ref_d_dft, model_name, sigma=None):
    """Runs the algorithm. If sigma is None, it estimates it based on model constraints."""
    trans_d = transform(test_d, ref_d_dft, 'residual_noise')
    
    if model_name == 'bm3d':
        rgb_vol, v_min, v_max = encode_voxel_to_rgb_global(trans_d)
        
        # Estimate sigma AFTER normalization for BM3D 
        # We pass the 3D volume of a single channel to get a global estimate
        if sigma is None:
            sigma = estimate_sigma(rgb_vol[:, :, :, 0], channel_axis=None)
            print(f"  -> [BM3D] Estimated Sigma on normalized data: {sigma:.6f}")
        
        denoised_rgb = np.zeros_like(rgb_vol)
        for i in range(trans_d.shape[0]):
            denoised_gray = bm3d(rgb_vol[i, :, :, 0], sigma_psd=sigma)
            denoised_rgb[i] = np.stack([denoised_gray]*3, axis=-1)
        denoised_trans = decode_rgb_to_voxel_global(denoised_rgb, v_min, v_max)

    elif model_name == 'bm4d':
        # Estimate sigma on the raw transformed 3D matrix for BM4D
        if sigma is None:
            sigma = estimate_sigma(trans_d, channel_axis=None)
            print(f"  -> [BM4D] Estimated Sigma on VST data: {sigma:.6f}")

        denoised_trans = bm4d(trans_d, sigma_psd=sigma)
        
    else:
        raise ValueError(f"Unknown model_name: {model_name}")

    return inverse_transform(denoised_trans, ref_d_dft, 'residual_noise')

# ==========================================
# 3. MAIN EXECUTION PIPELINE
# ==========================================
def main():
    print("Loading Base Data & DFT Reference...")
    with h5py.File(ref_path, 'r') as file:
        ref_d_mean = file['density'][:]
        ref_d_mean = ref_d_mean * (8.0 / np.sum(ref_d_mean))

    with h5py.File(dft_path, 'r') as file:
        dft_d = file['density'][:]
        dft_d = dft_d * (8.0 / np.sum(dft_d))
        
    noisy_files = sorted(glob.glob(os.path.join(base_dir, "density_tot_vmc_mean*.h5")))
    
    # Target specific sample
    target_sample = 6881280
    try:
        target_file = [f for f in noisy_files if f"{target_sample:010d}.h5" in f or f"_{target_sample}.h5" in f][0]
    except IndexError:
        raise FileNotFoundError(f"Could not find a file matching sample {target_sample} in {base_dir}")
    
    print(f"Selected sample count: {target_sample}")
    print(f"Loading file: {target_file}")
    
    with h5py.File(target_file, 'r') as file:
        test_d = file['density'][:]
        
    print("\n--- Running BM3D ---")
    denoised_d_bm3d = run_bm_with_sigma(test_d, dft_d, 'bm3d', sigma=None)
    js_bm3d = evaluate_and_enforce(denoised_d_bm3d, dft_d)
    print(f"BM3D JS Divergence: {js_bm3d:.8e}")

    print("\n--- Running BM4D ---")
    denoised_d_bm4d = run_bm_with_sigma(test_d, dft_d, 'bm4d', sigma=None)
    js_bm4d = evaluate_and_enforce(denoised_d_bm4d, dft_d)
    print(f"BM4D JS Divergence: {js_bm4d:.8e}")

if __name__ == "__main__":
    main()

Loading Base Data & DFT Reference...
Selected sample count: 6881280
Loading file: /pscratch/sd/k/kberard/SCGSR/Data/diamond_1x1x1_bfd/density_data/vmc_J2/density_tot_vmc_mean_0006881280.h5

--- Running BM3D ---
  -> [BM3D] Estimated Sigma on normalized data: 0.093641
BM3D JS Divergence: 1.28037751e-04

--- Running BM4D ---
  -> [BM4D] Estimated Sigma on VST data: 0.000383
BM4D JS Divergence: 1.26489950e-04


In [3]:
#!/usr/bin/env python3
import os
import glob
import re
import numpy as np
import h5py
from skimage.restoration import estimate_sigma

from bm3d import bm3d
from bm4d import bm4d

# ==========================================
# 0. CONFIGURATION 
# ==========================================
base_dir = "/pscratch/sd/k/kberard/SCGSR/Data/diamond_1x1x1_bfd/density_data/vmc_J2"
ref_path = os.path.join(base_dir, "density_tot_ref_mean.h5")
dft_path = '/global/u2/k/kberard/SCGSR/Research/Diamond/Data/density_tot_ref.h5'

# ==========================================
# 1. UNIFIED MATH & UTILITY FUNCTIONS
# ==========================================
def D_JS(p1, p2, tol=1e-16):
    """Calculates Jensen-Shannon Divergence."""
    p1 = p1 / (np.sum(p1) + 1e-16)
    p2 = p2 / (np.sum(p2) + 1e-16)
    pm = (p1 + p2) / 2
    p1_nonzero, p2_nonzero = np.abs(p1) > tol, np.abs(p2) > tol
    
    d = 0.5 * (
        (np.abs(p1[p1_nonzero]) * np.log(np.abs(p1[p1_nonzero]) / np.abs(pm[p1_nonzero]))).sum() + 
        (np.abs(p2[p2_nonzero]) * np.log(np.abs(p2[p2_nonzero]) / np.abs(pm[p2_nonzero]))).sum()
    )
    return d / np.log(2)

def encode_voxel_to_rgb_global(vol_3d):
    """Normalizes the volume globally to [0,1] and casts to 3-channel layout."""
    v_min, v_max = float(vol_3d.min()), float(vol_3d.max())
    if v_max == v_min: v_max = v_min + 1e-6
    normed = (vol_3d - v_min) / (v_max - v_min)
    return np.stack([normed]*3, axis=-1).astype(np.float32), v_min, v_max

def decode_rgb_to_voxel_global(rgb_volume, v_min, v_max):
    """Reverts the [0,1] normalization back to original raw scale."""
    return rgb_volume[:, :, :, 0] * (v_max - v_min) + v_min

def evaluate_and_enforce(denoised_d, ref_d):
    denoised_d = np.maximum(denoised_d, 0.0)
    denoised_d = denoised_d * (8.0 / (np.sum(denoised_d) + 1e-16))
    return D_JS(denoised_d, ref_d)

# ==========================================
# 2. INFERENCE DISPATCHER (NO VST ABLATION)
# ==========================================
def run_bm_no_vst(test_d, model_name, sigma=None):
    """Runs the algorithm directly on untransformed data."""
    
    if model_name == 'bm3d':
        # 1. Normalize the raw matrix to [0,1] and create 3 channels
        rgb_vol, v_min, v_max = encode_voxel_to_rgb_global(test_d)
        
        # 2. Estimate sigma on the normalized untransformed data
        if sigma is None:
            sigma = estimate_sigma(rgb_vol[:, :, :, 0], channel_axis=None)
            print(f"  -> [BM3D] Estimated Sigma on raw normalized data: {sigma:.6f}")
        
        # 3. Apply slice-by-slice BM3D denoising
        denoised_rgb = np.zeros_like(rgb_vol)
        for i in range(test_d.shape[0]):
            denoised_gray = bm3d(rgb_vol[i, :, :, 0], sigma_psd=sigma)
            denoised_rgb[i] = np.stack([denoised_gray]*3, axis=-1)
            
        # 4. Decode back to the raw untransformed density scale
        denoised_output = decode_rgb_to_voxel_global(denoised_rgb, v_min, v_max)

    elif model_name == 'bm4d':
        # 1. Estimate sigma directly on the raw, untransformed 3D matrix
        if sigma is None:
            sigma = estimate_sigma(test_d, channel_axis=None)
            print(f"  -> [BM4D] Estimated Sigma on raw 3D matrix: {sigma:.6f}")

        # 2. Run BM4D directly on the raw volume
        denoised_output = bm4d(test_d, sigma_psd=sigma)
        
    else:
        raise ValueError(f"Unknown model_name: {model_name}")

    return denoised_output

# ==========================================
# 3. MAIN EXECUTION PIPELINE
# ==========================================
def main():
    print("Loading Base Data & DFT Reference...")
    with h5py.File(ref_path, 'r') as file:
        ref_d_mean = file['density'][:]
        ref_d_mean = ref_d_mean * (8.0 / np.sum(ref_d_mean))

    with h5py.File(dft_path, 'r') as file:
        dft_d = file['density'][:]
        dft_d = dft_d * (8.0 / np.sum(dft_d))
        
    noisy_files = sorted(glob.glob(os.path.join(base_dir, "density_tot_vmc_mean*.h5")))
    
    # Target specific sample
    target_sample = 6881280
    try:
        target_file = [f for f in noisy_files if f"{target_sample:010d}.h5" in f or f"_{target_sample}.h5" in f][0]
    except IndexError:
        raise FileNotFoundError(f"Could not find a file matching sample {target_sample} in {base_dir}")
    
    print(f"Selected sample count: {target_sample}")
    print(f"Loading file: {target_file}")
    
    with h5py.File(target_file, 'r') as file:
        test_d = file['density'][:]
        
    print("\n--- Running BM3D (No VST) ---")
    denoised_d_bm3d = run_bm_no_vst(test_d, 'bm3d', sigma=None)
    js_bm3d = evaluate_and_enforce(denoised_d_bm3d, dft_d)
    print(f"BM3D (No VST) JS Divergence: {js_bm3d:.8e}")

    print("\n--- Running BM4D (No VST) ---")
    denoised_d_bm4d = run_bm_no_vst(test_d, 'bm4d', sigma=None)
    js_bm4d = evaluate_and_enforce(denoised_d_bm4d, dft_d)
    print(f"BM4D (No VST) JS Divergence: {js_bm4d:.8e}")

if __name__ == "__main__":
    main()

Loading Base Data & DFT Reference...
Selected sample count: 6881280
Loading file: /pscratch/sd/k/kberard/SCGSR/Data/diamond_1x1x1_bfd/density_data/vmc_J2/density_tot_vmc_mean_0006881280.h5

--- Running BM3D (No VST) ---
  -> [BM3D] Estimated Sigma on raw normalized data: 0.013832
BM3D (No VST) JS Divergence: 3.71715782e-04

--- Running BM4D (No VST) ---
  -> [BM4D] Estimated Sigma on raw 3D matrix: 0.000002
BM4D (No VST) JS Divergence: 2.21616865e-04


In [4]:
##################  estimate from noise model **********************

#!/usr/bin/env python3
import os
import glob
import numpy as np
import h5py

from bm3d import bm3d
from bm4d import bm4d

# ==========================================
# 0. CONFIGURATION 
# ==========================================
base_dir = "/pscratch/sd/k/kberard/SCGSR/Data/diamond_1x1x1_bfd/density_data/vmc_J2"
ref_path = os.path.join(base_dir, "density_tot_ref_mean.h5")
dft_path = '/global/u2/k/kberard/SCGSR/Research/Diamond/Data/density_tot_ref.h5'

# ==========================================
# 1. UNIFIED MATH & UTILITY FUNCTIONS
# ==========================================
def D_JS(p1, p2, tol=1e-16):
    """Calculates Jensen-Shannon Divergence."""
    p1 = p1 / (np.sum(p1) + 1e-16)
    p2 = p2 / (np.sum(p2) + 1e-16)
    pm = (p1 + p2) / 2
    p1_nonzero, p2_nonzero = np.abs(p1) > tol, np.abs(p2) > tol
    
    d = 0.5 * (
        (np.abs(p1[p1_nonzero]) * np.log(np.abs(p1[p1_nonzero]) / np.abs(pm[p1_nonzero]))).sum() + 
        (np.abs(p2[p2_nonzero]) * np.log(np.abs(p2[p2_nonzero]) / np.abs(pm[p2_nonzero]))).sum()
    )
    return d / np.log(2)

def transform(density, density_ref, transform_type='residual_noise'):
    """Applies the Variance Stabilizing Transformation (VST)."""
    if transform_type == 'residual_noise':
        return (density - density_ref) / (np.sqrt(np.abs(density_ref)) + 1e-12)
    return density

def inverse_transform(density_trans, density_ref, transform_type='residual_noise'):
    """Inverts the VST back to physical density."""
    if transform_type == 'residual_noise':
        return density_ref + (np.sqrt(np.abs(density_ref)) * density_trans)
    return density_trans

def encode_voxel_to_rgb_global(vol_3d):
    """Normalizes the volume globally to [0,1] and casts to 3-channel layout."""
    v_min, v_max = float(vol_3d.min()), float(vol_3d.max())
    if v_max == v_min: v_max = v_min + 1e-6
    normed = (vol_3d - v_min) / (v_max - v_min)
    return np.stack([normed]*3, axis=-1).astype(np.float32), v_min, v_max

def decode_rgb_to_voxel_global(rgb_volume, v_min, v_max):
    """Reverts the [0,1] normalization back to original raw scale."""
    return rgb_volume[:, :, :, 0] * (v_max - v_min) + v_min

def evaluate_and_enforce(denoised_d, ref_d):
    denoised_d = np.maximum(denoised_d, 0.0)
    denoised_d = denoised_d * (8.0 / (np.sum(denoised_d) + 1e-16))
    return D_JS(denoised_d, ref_d)

# ==========================================
# 2. INFERENCE DISPATCHER (ANALYTICAL SIGMA)
# ==========================================
def run_bm_analytical(test_d, ref_d_dft, model_name, N_samples):
    """
    Runs the algorithm using the exact mathematical sigma 
    derived from the sample count N.
    """
    trans_d = transform(test_d, ref_d_dft, 'residual_noise')
    
    # 1. Base analytical standard deviation of the VST space
    sigma_vst = 1.0 / np.sqrt(N_samples)
    
    if model_name == 'bm3d':
        rgb_vol, v_min, v_max = encode_voxel_to_rgb_global(trans_d)
        
        # 2. Scale the analytical sigma by the [0, 1] dynamic range normalization
        sigma_bm3d = sigma_vst / (v_max - v_min)
        print(f"  -> [BM3D] Base VST Sigma: {sigma_vst:.6e} | Normalized Sigma: {sigma_bm3d:.6e}")
        
        denoised_rgb = np.zeros_like(rgb_vol)
        for i in range(trans_d.shape[0]):
            denoised_gray = bm3d(rgb_vol[i, :, :, 0], sigma_psd=sigma_bm3d)
            denoised_rgb[i] = np.stack([denoised_gray]*3, axis=-1)
        denoised_trans = decode_rgb_to_voxel_global(denoised_rgb, v_min, v_max)

    elif model_name == 'bm4d':
        # 2. BM4D processes the VST data directly on the raw scale
        print(f"  -> [BM4D] Applied Analytical Sigma: {sigma_vst:.6e}")
        denoised_trans = bm4d(trans_d, sigma_psd=sigma_vst)
        
    else:
        raise ValueError(f"Unknown model_name: {model_name}")

    return inverse_transform(denoised_trans, ref_d_dft, 'residual_noise')

# ==========================================
# 3. MAIN EXECUTION PIPELINE
# ==========================================
def main():
    print("Loading Base Data & DFT Reference...")
    with h5py.File(ref_path, 'r') as file:
        ref_d_mean = file['density'][:]
        ref_d_mean = ref_d_mean * (8.0 / np.sum(ref_d_mean))

    with h5py.File(dft_path, 'r') as file:
        dft_d = file['density'][:]
        dft_d = dft_d * (8.0 / np.sum(dft_d))
        
    noisy_files = sorted(glob.glob(os.path.join(base_dir, "density_tot_vmc_mean*.h5")))
    
    # Target specific sample / MC count
    target_sample = 6881280
    try:
        target_file = [f for f in noisy_files if f"{target_sample:010d}.h5" in f or f"_{target_sample}.h5" in f][0]
    except IndexError:
        raise FileNotFoundError(f"Could not find a file matching sample {target_sample} in {base_dir}")
    
    print(f"Selected file: {target_file}")
    print(f"MC Samples (N): {target_sample}")
    
    with h5py.File(target_file, 'r') as file:
        test_d = file['density'][:]
        
    print("\n--- Running BM3D (Analytical) ---")
    denoised_d_bm3d = run_bm_analytical(test_d, dft_d, 'bm3d', N_samples=target_sample)
    js_bm3d = evaluate_and_enforce(denoised_d_bm3d, dft_d)
    print(f"BM3D JS Divergence: {js_bm3d:.8e}")

    print("\n--- Running BM4D (Analytical) ---")
    denoised_d_bm4d = run_bm_analytical(test_d, dft_d, 'bm4d', N_samples=target_sample)
    js_bm4d = evaluate_and_enforce(denoised_d_bm4d, dft_d)
    print(f"BM4D JS Divergence: {js_bm4d:.8e}")

if __name__ == "__main__":
    main()

Loading Base Data & DFT Reference...
Selected file: /pscratch/sd/k/kberard/SCGSR/Data/diamond_1x1x1_bfd/density_data/vmc_J2/density_tot_vmc_mean_0006881280.h5
MC Samples (N): 6881280

--- Running BM3D (Analytical) ---
  -> [BM3D] Base VST Sigma: 3.812110e-04 | Normalized Sigma: 9.308421e-02
BM3D JS Divergence: 1.28333448e-04

--- Running BM4D (Analytical) ---
  -> [BM4D] Applied Analytical Sigma: 3.812110e-04
BM4D JS Divergence: 1.26598601e-04


In [5]:
1/ np.sqrt(6881280)

np.float64(0.0003812109659955208)